In [4]:
import tarfile

with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall(filter='fully_trusted')

In [63]:
import pyprind
import pandas as pd
import os
import sys

basepath = 'aclImdb'
labels = {'pos': 1, 'neg': 0}
data_list = []
pbar = pyprind.ProgBar(50000, stream=sys.stdout)
for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
            data_list.append([txt, labels[l]])
            pbar.update()
df = pd.DataFrame(data_list, columns=['review', 'sentiment'])

In [77]:
import numpy as np

df = pd.DataFrame(data_list, columns=['review', 'sentiment'])
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')

In [79]:
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={0: 'review', 1: 'sentiment'})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


In [80]:
df.shape

(50000, 2)

In [32]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()
docs = np.array(["The sun is shining",
                 'The weather is sweet',
                 'The sun is shining, the weather is sweet,'
                 'and one and one is two'])
bag = count.fit_transform(docs)

In [33]:
print(count.vocabulary_)

{'the': 6, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 8, 'sweet': 5, 'and': 0, 'one': 2, 'two': 7}


In [34]:
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


In [41]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer(use_idf=True,
                         norm='l2',
                         smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs)).toarray())

[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


In [54]:
import math

count_array = bag.toarray()
tf_idf_list = []


def count_sum(index):
    counter = 0
    for i in range(count_array.shape[0]):
        if count_array[i][index] != 0:
            counter += 1
    return counter


for i_sym in range(count_array.shape[1]):
    i_sym_count = count_sum(i_sym)
    idf = math.log((1 + 3) / (1 + i_sym_count))
    tf_idf_list.append(count_array[2][i_sym] * (idf + 1))
print(tf_idf_list)


def l2_normalize(some_list):
    summ = 0
    for item in some_list:
        summ += math.pow(item, 2)
    normalized_list = [round(item / math.sqrt(summ), 2) for item in some_list]
    return normalized_list


tf_idf_list = l2_normalize(tf_idf_list)
print(tf_idf_list)

[np.float64(3.386294361119891), np.float64(3.0), np.float64(3.386294361119891), np.float64(1.2876820724517808), np.float64(1.2876820724517808), np.float64(1.2876820724517808), np.float64(2.0), np.float64(1.6931471805599454), np.float64(1.2876820724517808)]
[np.float64(0.5), np.float64(0.45), np.float64(0.5), np.float64(0.19), np.float64(0.19), np.float64(0.19), np.float64(0.3), np.float64(0.25), np.float64(0.19)]


In [82]:
df.loc[0, 'review'][-50:]


'is seven.<br /><br />Title (Brazil): Not Available'

In [86]:
import re


def preprocessor(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    return text

In [83]:
preprocessor(df.loc[0, 'review'][-50:])

'is seven title brazil not available'

In [87]:
preprocessor("</a>This :) is :( a test :-)!")

'this is a test :) :( :)'

In [88]:
df['review'] = df['review'].apply(preprocessor)

In [90]:
from nltk.stem.porter import PorterStemmer

porter = PorterStemmer()


def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]


tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

In [97]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\perev\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [98]:
from nltk.corpus import stopwords

stop = stopwords.words('english')
[w for w in tokenizer_porter('a runner likes running and runs a lot') if w not in stop]

['runner', 'like', 'run', 'run', 'lot']

In [132]:
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
print(X_test)
y_test = df.loc[25000:, 'sentiment'].values

['there s a part of me that would like to give this movie a high rating considering that it was made in 1953 this is a very courageous movie about transvestites tackling the issue fairly seriously and sympathetically and offering the viewer a lot of information on the subject and trying very hard not to stereotype the movie clearly makes the point that transvestites are not homosexuals and that aside from wearing women s clothing they lead a relatively normal life it deals with the pain of not being accepted in society the plot revolves around a police officer lyle talbot desperately trying to understand the issue because of the recent suicide of a transvestite so you have to give everyone involved with this movie credit for taking on such a controversial in the context of 1953 subject having said all that i m also sorry to say that this movie is absolutely dreadful in trying to portray glen glenda s edward d wood pain the movie falls into silly and at times surprisingly again given th

In [101]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None,
                        token_pattern=None)
small_param_grid = [
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [None],
        'vect__tokenizer': [tokenizer_porter],
        'clf__l1_ratio': [0],
        'clf__C': [1., 10.]
    },
    {
        'vect__ngram_range': [(1, 1)],
        'vect__stop_words': [stop, None],
        'vect__tokenizer': [tokenizer_porter],
        'vect__use_idf': [False],
        'vect__norm': [None],
        'clf__l1_ratio': [0],
        'clf__C': [1., 10.]
    }
]
lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear'))
])
gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid, cv=5, n_jobs=-1, verbose=2)
gs_lr_tfidf.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Users\perev\PythonProject\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf__C': [1.0, 10.0], 'clf__l1_ratio': [0], 'vect__ngram_range': [(1, ...)], 'vect__stop_words': [None], ...}, {'clf__C': [1.0, 10.0], 'clf__l1_ratio': [0], 'vect__ngram_range': [(1, ...)], 'vect__norm': [None], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the

In [103]:
print(f'Точность CV: {gs_lr_tfidf.best_score_:.3f}')

Точность CV: 0.892


In [104]:
clf = gs_lr_tfidf.best_estimator_
print(f'Точность на тестовом наборе: {clf.score(X_test, y_test):.3f}')

Точность на тестовом наборе: 0.894


In [134]:
import numpy as np
import re
from nltk.corpus import stopwords

stop = stopwords.words('english')


def tokenizer(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    tokenized = [w for w in text.split() if w not in stop]
    return tokenized

def stream_docs(path):
    with open(path, 'r', encoding='utf-8') as csv:
        next(csv)
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label



In [135]:
def get_minibatch(doc_stream, size):
    docs, y = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            y.append(label)
    except StopIteration:
        return None, None
    return docs, y

In [136]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
vect = HashingVectorizer(decode_error='ignore',
                         n_features=2**21,
                         preprocessor=None,
                         tokenizer=tokenizer)
clf = SGDClassifier(loss='log_loss', random_state=1)
doc_stream = stream_docs(path='movie_data.csv')

In [137]:
import pyprind
pbar = pyprind.ProgBar(45)
classes = np.array([0, 1])
for _ in range(45):
    X_train, y_train = get_minibatch(doc_stream, size=1000)
    if not X_train:
        continue
    else:
        X_train = vect.transform(X_train)
        clf.partial_fit(X_train, y_train, classes=classes)
    pbar.update()

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:21


In [138]:
X_test, y_test = get_minibatch(doc_stream, size=5000)
X_test = vect.transform(X_test)
print(f'Точность: {clf.score(X_test, y_test):.3f}')

Точность: 0.868


In [139]:
clf = clf.partial_fit(X_test, y_test)

In [141]:
import pandas as pd
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={'0': 'review', '1': 'sentiment'})


In [143]:
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer(stop_words='english',
                        max_df=0.1,
                        max_features=5000,
                        )
X = count.fit_transform(df['review'].values)

In [144]:
from sklearn.decomposition import LatentDirichletAllocation
lda = LatentDirichletAllocation(n_components=10,
                                random_state=123,
                                learning_method='batch',)
X_topics = lda.fit_transform(X)

In [147]:
lda.components_.shape

(10, 5000)

In [148]:
n_top_words = 5
feature_names = count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    print(f'Тема {topic_idx + 1}:')
    print(' '.join([feature_names[i]
                    for i in topic.argsort()\
                    [:-n_top_words - 1:-1]]))

Тема 1:
horror worst script effects budget
Тема 2:
dvd watched video music guy
Тема 3:
war american series history documentary
Тема 4:
game killer murder thriller crime
Тема 5:
kids comedy episode series school
Тема 6:
family woman mother beautiful feel
Тема 7:
role performance comedy john plays
Тема 8:
action horror john effects dr
Тема 9:
book version original read music
Тема 10:
action wife father police james
